# Submit từ toàn bộ dữ liệu đã xử lý — giữ URL cũ

**Chạy trên Colab, chọn T4 GPU → Run all.** Cell 2 mặc định chỉ khôi phục
checkpoint bằng manifest/báo cáo JSON. Không mở lại corpus, vector hoặc catalog/BM25.
Nếu báo cáo thiếu hoặc không khớp, cell 2 dừng; không tự chạy preparation lại.
Với dữ liệu mới chưa chuẩn bị, chủ động bật `ALLOW_CPU_PREPARATION` ở cell 2
và chạy CPU trước. Khi có `CPU_PREPARATION_COMPLETE`, tắt lại tùy chọn này,
đổi sang T4 GPU và Run all cùng notebook.
Đổi runtime mất biến/ổ tạm; các checkpoint đã publish trên Drive vẫn được giữ và tự nhận lại.
Khi đã có freeze receipt, resume bỏ qua vòng chọn/chép toàn bộ source part.
Spool local còn thì dùng lại; runtime mới chỉ khôi phục document/child cần để dựng spool,
chưa tải parent/section/ledger ở bước Index inputs. Log ghi riêng part reused/new.
Resume dùng cursor trên Drive để bỏ qua nội dung input part cũ. Runtime mới vẫn đọc source,
nhưng spool chỉ giữ hash sau cursor. SHA256 input cũ được kiểm khi bước tiếp theo dùng file.
Catalog tự dùng tối đa 4 CPU process, giữ nguyên PyVi/Jieba và BM25. Mỗi shard có
checkpoint tách từ trên Drive; SQLite tiếp tục từ shard đã commit nếu còn cùng runtime.
Nếu mất runtime khi catalog còn dở, phần SQL chưa có catalog hoàn tất cần phục hồi;
checkpoint tách từ vẫn dùng lại, không tính embedding lại.
Catalog đã hoàn tất của nguồn cũ được kế thừa khi receipt xác nhận toàn bộ nguồn vẫn còn:
giữ BM25/document/child/parent cũ, chỉ dựng lại ánh xạ dense position và index phần mới.
Log `Catalog reuse` và `Disk catalog/BM25 delta` ghi riêng phần cũ/mới; CPU vẫn cần cho delta.
Khôi phục catalog xác minh SHA ngay lúc giải nén. Bước kế thừa báo phase mỗi 15 giây,
lưu cursor chuyển ánh xạ theo khối 65.536 dòng; ngắt cùng runtime tiếp tục khối đã commit.
Không đọc/quick_check toàn catalog thêm lần nữa trong bước kế thừa. Catalog cuối vẫn được kiểm đầy đủ.
Chỉ cần chỉnh `DATA_ROOT_PATH` nếu Drive của tài khoản đang chạy dùng đường dẫn khác.

Giữ URL thuộc snapshot gốc, kể cả URL bị BTC loại trong bản dataset mới, theo yêu cầu
của Sếp. Không xóa dữ liệu gốc, không crawl/extract/chunk lại. Các query và ID gắn
đúng snapshot nguồn; lượt này **chưa xác nhận tương thích phạm vi BTC mới**.

Trên GPU: bổ sung BGE còn thiếu → lưu ZIP BGE/BM25 đủ 1.200 query trước →
dịch EN/ZH + PICO/subqueries/HyDE + document dense + Qwen rerank hai tầng +
source parents → ZIP nâng cấp. Qwen child dense dùng khi cache đầy đủ;
checkpoint Qwen chưa đủ vẫn được giữ. Đây chưa phải hoàn tất toàn bộ master plan.
Mỗi model ≤15B. Chưa có nhãn độc lập để fine-tune/hiệu chỉnh hoặc cam kết điểm.

Mỗi thời điểm chỉ **một phiên notebook này** ghi cùng DATA_ROOT. Các stage, vector,
model input, query và ZIP lưu bền trên Drive. Nếu bị ngắt, Run all cùng notebook
và cấu hình để tiếp tục checkpoint. Khi đã có ZIP baseline, có thể nộp thử trong
lúc lượt strong còn đang chạy; ZIP mạnh hơn chỉ xuất sau khi đủ 1.200 query.

In [ ]:
#@title 1. Drive, code và môi trường (tự nhận CPU/GPU)
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys

DATA_ROOT_PATH = "/content/drive/MyDrive/VietMedBridge/data" #@param {type:"string"}
DATA_ROOT = Path(DATA_ROOT_PATH)
WORK_DIR = Path("/content/vmb_submit_work")
CHECKOUT = Path("/content/VietMedBridge-submit")
REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
WORKFLOW_VERSION = "all-available-legacy-20261017-v9"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK_DIR.mkdir(parents=True,exist_ok=True)

if not Path("/var/colab/hostname").exists():
    raise RuntimeError("Notebook này chạy trên Google Colab. Kaggle dùng notebook embedding worker riêng.")
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    (DATA_ROOT / "active_data_candidate.json").read_bytes()
except OSError as error:
    if error.errno != 107:
        raise
    print("Drive preflight: remount trước khi đọc code lock hoặc nạp model.",flush=True)
    drive.mount("/content/drive",force_remount=True)
    (DATA_ROOT / "active_data_candidate.json").read_bytes()
if not (DATA_ROOT / "active_data_candidate.json").is_file():
    raise RuntimeError("DATA_ROOT chưa trỏ tới data đã freeze. Chỉnh DATA_ROOT ở đầu cell, không tạo data rỗng.")
print("DATA_ROOT:", DATA_ROOT,flush=True)
# A lock for this workflow, separate from old notebook/data-processing locks.
code_lock = DATA_ROOT / "retrieval" / (WORKFLOW_VERSION+"-code-lock.json")
locked = json.loads(code_lock.read_text()) if code_lock.exists() else None
if locked and (locked.get("repo") != REPO_URL or not re.fullmatch(r"[0-9a-f]{40}",locked.get("commit",""))):
    raise ValueError("Submission code lock không hợp lệ.")
reference = locked["commit"] if locked else "origin/main"
def command(args):
    print("Setup:"," ".join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),check=True)
if not (CHECKOUT / ".git").is_dir():
    command(["git","clone",REPO_URL,CHECKOUT])
command(["git","-C",CHECKOUT,"fetch","origin"])
command(["git","-C",CHECKOUT,"checkout","--detach",reference])
COMMIT = subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
if not (CHECKOUT / "src/vietmedbridge/submission_workflow.py").is_file():
    raise RuntimeError("Checkout chưa có workflow submission mới. Dùng notebook cùng bản code đã push.")
# Detect the physical GPU before importing torch/transformers.
GPU_SESSION = False
if shutil.which("nvidia-smi"):
    probe = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True)
    GPU_SESSION = probe.returncode == 0 and bool(probe.stdout.strip())
    print("Physical GPUs:",probe.stdout.strip() or "none",flush=True)
runtime = None
runtime_lock = DATA_ROOT / "retrieval_embedding_runtime_lock.json"
if GPU_SESSION:
    if not runtime_lock.is_file():
        raise RuntimeError("Thiếu runtime lock của vector đã nhập. Khôi phục retrieval_embedding_runtime_lock.json trước khi inference.")
    lock = json.loads(runtime_lock.read_text())
    payload = {k:v for k,v in lock.items() if k != "manifest_sha256"}
    checksum = hashlib.sha256(json.dumps(payload,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()
    if checksum != lock.get("manifest_sha256"):
        raise ValueError("Embedding runtime lock đã thay đổi.")
    runtime = lock["runtime"]
    if (set(runtime) != {"torch","transformers","bitsandbytes"}
        or any(not re.fullmatch(r"[0-9][A-Za-z0-9.+_-]*",v) for v in runtime.values())
        or not re.search(r"\+cu[0-9]+$",runtime["torch"])):
        raise ValueError("Runtime lock cần CUDA torch, transformers và bitsandbytes hợp lệ.")
    for name,expected in lock["model_code"].items():
        if Path(name).name != name or hashlib.sha256((CHECKOUT / "src/vietmedbridge" / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Model producer code khác cache đã tính; dừng trước khi nạp model.")
    if "torch" in sys.modules and sys.modules["torch"].__version__ != runtime["torch"]:
        raise RuntimeError("Runtime đang giữ torch phiên bản cũ. Restart session rồi Run all; giữ nguyên Drive/checkpoint.")
    try:
        torch_version = importlib.metadata.version("torch")
    except importlib.metadata.PackageNotFoundError:
        torch_version = None
    if torch_version != runtime["torch"]:
        suffix = re.search(r"\+(cu[0-9]+)$",runtime["torch"])[1]
        command([sys.executable,"-m","pip","install","torch=="+runtime["torch"],"--index-url","https://download.pytorch.org/whl/"+suffix])
        command([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"])
    constraints = WORK_DIR / "runtime-constraints.txt"
    constraints.write_text("".join(f"{name}=={version}\n" for name,version in runtime.items()))
    command([sys.executable,"-m","pip","install","-c",constraints,"-e",str(CHECKOUT)+"[retrieval,strong]"])
else:
    command([sys.executable,"-m","pip","install","-e",CHECKOUT,
        "numpy>=1.26,<3","faiss-cpu==1.12.0","pyvi==0.1.1","jieba==0.42.1",
        "scikit-learn==1.7.2","sklearn-crfsuite==0.5.0"])
sys.path.insert(0,str(CHECKOUT / "src"))
import vietmedbridge
if Path(vietmedbridge.__file__).resolve() != (CHECKOUT / "src/vietmedbridge/__init__.py").resolve():
    raise RuntimeError("Kernel đang giữ package của checkout khác. Restart session rồi Run all.")
importlib.invalidate_caches()
for module_name in ("vietmedbridge.artifacts","vietmedbridge.colab_storage","vietmedbridge.content_embeddings",
                    "vietmedbridge.input_resume","vietmedbridge.input_spool","vietmedbridge.index_inputs",
                    "vietmedbridge.catalog_reuse","vietmedbridge.catalog_builder","vietmedbridge.catalog_storage",
                    "vietmedbridge.official_scope","vietmedbridge.submission_workflow"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
from vietmedbridge.artifacts import atomic_json, read_json, sha256_file, verify_file
if not locked:
    atomic_json(code_lock,{"repo":REPO_URL,"commit":COMMIT,"workflow":WORKFLOW_VERSION})
try:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass # Public models do not require a token.
if GPU_SESSION:
    import torch
    if not torch.cuda.is_available() or torch.__version__ != runtime["torch"]:
        raise RuntimeError("CUDA torch chưa hoạt động. Restart session rồi Run all sau khi setup đã cài xong.")
    print("CUDA:",torch.__version__,torch.cuda.get_device_name(0),flush=True)
print("Code:",COMMIT,"| package:",vietmedbridge.__file__,flush=True)
print("Phase:","GPU inference" if GPU_SESSION else "CPU corpus/catalog preparation",flush=True)


In [ ]:
#@title 2. Chỉ khôi phục checkpoint JSON; không tự chạy preparation lại
ALLOW_CPU_PREPARATION = False #@param {type:"boolean"}
"""Restore a sealed submission handoff using JSON only; never start CPU preparation.

Embedded in the notebook so old producer/code locks and corpus fingerprints stay
unchanged. This checks identities, not catalog/vector payload integrity. Native
readers still validate payloads when inference actually consumes them.
"""
from pathlib import Path

from vietmedbridge.artifacts import atomic_json, digest_json, read_json
from vietmedbridge.official_scope import discover_sources
from vietmedbridge.scale_benchmark import load_handoff


def restore_submission_preparation_only(data_root, checkout):
    root, checkout = Path(data_root), Path(checkout)
    print("RESUME_ONLY: đọc manifest/báo cáo JSON; không mở corpus, vector hoặc catalog/BM25.", flush=True)
    build, candidate, inputs = load_handoff(root)
    config_sha = digest_json(read_json(checkout / "configs/retrieval_full.json"))
    source_names = discover_sources(root)
    corpus_sha = read_json(root / "raw/snapshot.json")["files"]["links_corpus.parquet"]["sha256"]
    identity = {"snapshot_sha256": candidate["snapshot_sha256"],
        "candidate_manifest_sha256": candidate["candidate_manifest_sha256"],
        "index_inputs_manifest_sha256": inputs["manifest_sha256"]}
    pointer = root / "retrieval/current_submission.json"
    reports = [pointer, build / "submission_preparation.json",
        root / "retrieval/cpu_preparation" / (candidate["candidate_manifest_sha256"][:16] + ".json")]
    for path in reports:
        if not path.is_file():
            continue
        saved = read_json(path)
        if (saved.get("state") != "CPU_PREPARATION_COMPLETE"
            or saved.get("build_run") != build.name
            or saved.get("candidate_manifest_sha256") != candidate["candidate_manifest_sha256"]
            or saved.get("retrieval_config_sha256") != config_sha
            or saved.get("official_sha256") != candidate["origin_corpus_sha256"]
            or candidate.get("input_kind") != "AVAILABLE_CORPUS"
            or saved.get("legacy_urls_retained") is not True
            or saved.get("source_builds") != source_names
            or any(saved.get("catalog", {}).get(k) != v for k, v in identity.items())
            or corpus_sha != candidate["origin_corpus_sha256"]):
            continue
        result = saved | {"results_root": str(root.parent / "submission_runs" / build.name)}
        if not pointer.is_file() or read_json(pointer) != result:
            atomic_json(pointer, result)  # only a small handoff pointer; no stage output is rebuilt
        print(f"RESUME_READY: {build.name} | {saved['documents']:,} documents | CPU preparation reused.", flush=True)
        return result
    raise RuntimeError(
        "RESUME_ONLY_STOP: thiếu hoặc không khớp báo cáo CPU_PREPARATION_COMPLETE. "
        "Chưa đọc lại corpus/vector/catalog và chưa bắt đầu preparation. "
        "Kiểm tra DATA_ROOT/candidate/code lock; chỉ bật ALLOW_CPU_PREPARATION khi chủ động chuẩn bị dữ liệu mới.")


if ALLOW_CPU_PREPARATION:
    from vietmedbridge.submission_workflow import ensure_submission_preparation
    PREPARATION = ensure_submission_preparation(DATA_ROOT,CHECKOUT,WORK_DIR,gpu_session=GPU_SESSION)
else:
    PREPARATION = restore_submission_preparation_only(DATA_ROOT,CHECKOUT)
print(json.dumps({k: PREPARATION[k] for k in (
    "state", "build_run", "candidate_manifest_sha256", "documents", "unique_dense_inputs"
)},ensure_ascii=False,indent=2),flush=True)
if GPU_SESSION:
    print("Tiếp tục cell 3. Cell 2 chưa kiểm payload; cell inference kiểm checkpoint khi sử dụng.",flush=True)
else:
    print("CPU handoff đã có. Chọn GPU rồi chạy cell 1 → 2 → 3; giữ ALLOW_CPU_PREPARATION=False khi resume.",flush=True)


In [ ]:
#@title 3. BGE phần còn thiếu và ZIP đầu tiên đủ 1.200 query (GPU)
"""Bounded parallel map checks and visible GPU startup; producer files stay pinned."""
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path
import sys
import threading
import time

from vietmedbridge.artifacts import atomic_json, digest_json, read_json, utc_now, verify_file
from vietmedbridge.content_embeddings import checked, seal
from vietmedbridge.portable_embeddings import resources_relative


def read_bge_maps_parallel(data_root, candidate, inputs, config, *, progress=print):
    """Same BGE identity/map/payload checks as imported_content_manifest, in frozen order."""
    view = Path(data_root) / resources_relative(candidate, config) / 'bge'
    path = view / 'config.json'
    if not path.is_file():
        return view, None
    identity = read_json(path)
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != 1024
        or any(identity.get('encoder', {}).get(k) != v for k, v in config['dense'].items())):
        raise ValueError('Imported BGE view differs from the active frozen corpus/model.')
    signature = digest_json(identity)
    sources = inputs['parts']
    progress(f'BGE mapping check: 0/{len(sources):,} | up to 8 readers | SHA256 preserved')

    def check(source):
        marker = view / f"map-{source['start']:012d}.done.json"
        if not marker.is_file():
            return None
        saved = checked(read_json(marker))
        expected = {'signature': signature, 'start': source['start'], 'rows': source['rows'],
            'input_sha256': source['sha256'], 'path': f"map-{source['start']:012d}.parquet"}
        if any(saved.get(k) != v for k, v in expected.items()):
            raise ValueError('Imported BGE mapping differs from the frozen input order.')
        verify_file(view / saved['path'], saved['sha256'])
        return saved

    ordered = [None] * len(sources)
    found = 0
    pool = ThreadPoolExecutor(max_workers=min(8, max(1, len(sources))))
    try:
        futures = {pool.submit(check, source): i for i, source in enumerate(sources)}
        for count, future in enumerate(as_completed(futures), 1):
            saved = future.result()
            ordered[futures[future]] = saved
            found += saved is not None
            if count % 32 == 0 or count == len(sources):
                progress(f'BGE mapping check: {count:,}/{len(sources):,} | valid maps {found:,}')
    except BaseException:
        # Do not trap the notebook in executor shutdown if Drive readers stall.
        # In-flight work only reads files; it never publishes or changes checkpoints.
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    else:
        pool.shutdown(wait=True)
    parts = [p for p in ordered if p is not None]
    return view, seal(identity | {'signature': signature, 'parts': parts,
        'state': 'COMPLETE' if len(parts) == len(sources) else 'IN_PROGRESS'})


class SubmissionProgress:
    def __init__(self, data_root, work_dir, label, *, interval=15):
        self.root, self.work, self.label = Path(data_root), Path(work_dir), label
        self.interval, self.main_id = interval, threading.get_ident()
        self.started, self.last_saved = time.monotonic(), 0
        self.phase = 'Entering submission workflow'
        self.binding = {}
        self.stop = threading.Event()

    def location(self):
        frame = sys._current_frames().get(self.main_id)
        location = None
        while frame is not None:
            if frame.f_code.co_name in {'read_json', 'sha256_file'}:
                value = frame.f_locals.get('path')
                if isinstance(value, (str, Path)):
                    location = str(value)
                    break
            if 'vietmedbridge' in frame.f_code.co_filename:
                location = Path(frame.f_code.co_filename).name + ':' + str(frame.f_lineno)
            frame = frame.f_back
        return location

    def snapshot(self, state='RUNNING'):
        return {'state': state, 'variant': self.label, 'phase': self.phase,
            'elapsed_seconds': round(time.monotonic() - self.started, 1),
            'waiting_at': self.location(), 'updated_at': utc_now(),
            **self.binding,
            'scope': 'Live stage diagnostic; not an embedding completion or submission receipt'}

    def report(self, message):
        self.phase = message
        print(message, flush=True)
        if time.monotonic() - self.last_saved >= 15:
            try:
                atomic_json(self.root / 'retrieval/submission_progress.json', self.snapshot())
                self.last_saved = time.monotonic()
            except OSError:
                pass

    def heartbeat(self):
        while not self.stop.wait(self.interval):
            status = self.snapshot()
            print(f"SUBMISSION heartbeat | {status['elapsed_seconds']:.0f}s | {self.phase}"
                f" | waiting at {status['waiting_at']}", flush=True)
            try:
                atomic_json(self.work / 'submission_progress.json', status)
            except OSError:
                pass  # Diagnostics cannot replace or invalidate native checkpoint errors.


@contextmanager
def submission_gpu_progress(data_root, work_dir, *, label='baseline'):
    """Single-writer notebook adapter; restores every patched binding on exit."""
    from vietmedbridge import scale_baseline, submission_workflow
    progress = SubmissionProgress(data_root, work_dir, label)
    original_maps = scale_baseline.imported_bge_manifest
    names = ('_submission_context', '_saved_submission', '_release_catalog_staging', '_ensure_primary')
    originals = {name: getattr(submission_workflow, name) for name in names}

    def wrap(name, function):
        def observed(*args, **kwargs):
            progress.report('SUBMISSION stage: ' + name)
            result = function(*args, **kwargs)
            if name == '_submission_context':
                progress.binding = {'build_run': result[3].name,
                    'candidate_manifest_sha256': result[4]['candidate_manifest_sha256'],
                    'input_count': result[5]['input_count']}
            progress.report('SUBMISSION stage complete: ' + name)
            return result
        return observed

    scale_baseline.imported_bge_manifest = lambda *a, **k: read_bge_maps_parallel(*a, **k, progress=progress.report)
    for name, function in originals.items():
        setattr(submission_workflow, name, wrap(name, function))
    thread = threading.Thread(target=progress.heartbeat, name='submission-progress', daemon=True)
    progress.report('SUBMISSION startup | live progress every 15 seconds | same candidate and checkpoints')
    thread.start()
    state = 'STOPPED'
    try:
        yield progress
        state = 'CALL_FINISHED'
    finally:
        progress.stop.set()
        thread.join(timeout=1)
        scale_baseline.imported_bge_manifest = original_maps
        for name, function in originals.items():
            setattr(submission_workflow, name, function)
        status = progress.snapshot(state)
        try:
            atomic_json(Path(work_dir) / 'submission_progress.json', status)
        except OSError:
            pass
        try:
            atomic_json(Path(data_root) / 'retrieval/submission_progress.json', status)
        except OSError:
            pass


from vietmedbridge.submission_workflow import run_current_baseline
BASELINE = None
if GPU_SESSION:
    with submission_gpu_progress(DATA_ROOT, WORK_DIR, label="baseline"):
        BASELINE = run_current_baseline(DATA_ROOT,CHECKOUT,WORK_DIR,code_commit=COMMIT)
    print(json.dumps(BASELINE["ready"],ensure_ascii=False,indent=2),flush=True)
    if BASELINE["ready"]["state"] != "READY_FOR_MANUAL_UPLOAD":
        raise RuntimeError("Baseline chưa đủ query/index: Run all cùng cấu hình để resume. Chưa nạp Qwen reranker.")
    print("ĐÃ CÓ ZIP ĐẦU TIÊN trên Drive. Cell cuối có thể tải bản này trong lúc chuẩn bị bản mạnh hơn.",flush=True)
else:
    print("GPU phase skipped on CPU.")


In [ ]:
#@title 4. Bản nâng cấp: dịch + expansion + document dense + Qwen rerank + ZIP (GPU)
"""Bounded parallel map checks and visible GPU startup; producer files stay pinned."""
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path
import sys
import threading
import time

from vietmedbridge.artifacts import atomic_json, digest_json, read_json, utc_now, verify_file
from vietmedbridge.content_embeddings import checked, seal
from vietmedbridge.portable_embeddings import resources_relative


def read_bge_maps_parallel(data_root, candidate, inputs, config, *, progress=print):
    """Same BGE identity/map/payload checks as imported_content_manifest, in frozen order."""
    view = Path(data_root) / resources_relative(candidate, config) / 'bge'
    path = view / 'config.json'
    if not path.is_file():
        return view, None
    identity = read_json(path)
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != 1024
        or any(identity.get('encoder', {}).get(k) != v for k, v in config['dense'].items())):
        raise ValueError('Imported BGE view differs from the active frozen corpus/model.')
    signature = digest_json(identity)
    sources = inputs['parts']
    progress(f'BGE mapping check: 0/{len(sources):,} | up to 8 readers | SHA256 preserved')

    def check(source):
        marker = view / f"map-{source['start']:012d}.done.json"
        if not marker.is_file():
            return None
        saved = checked(read_json(marker))
        expected = {'signature': signature, 'start': source['start'], 'rows': source['rows'],
            'input_sha256': source['sha256'], 'path': f"map-{source['start']:012d}.parquet"}
        if any(saved.get(k) != v for k, v in expected.items()):
            raise ValueError('Imported BGE mapping differs from the frozen input order.')
        verify_file(view / saved['path'], saved['sha256'])
        return saved

    ordered = [None] * len(sources)
    found = 0
    pool = ThreadPoolExecutor(max_workers=min(8, max(1, len(sources))))
    try:
        futures = {pool.submit(check, source): i for i, source in enumerate(sources)}
        for count, future in enumerate(as_completed(futures), 1):
            saved = future.result()
            ordered[futures[future]] = saved
            found += saved is not None
            if count % 32 == 0 or count == len(sources):
                progress(f'BGE mapping check: {count:,}/{len(sources):,} | valid maps {found:,}')
    except BaseException:
        # Do not trap the notebook in executor shutdown if Drive readers stall.
        # In-flight work only reads files; it never publishes or changes checkpoints.
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    else:
        pool.shutdown(wait=True)
    parts = [p for p in ordered if p is not None]
    return view, seal(identity | {'signature': signature, 'parts': parts,
        'state': 'COMPLETE' if len(parts) == len(sources) else 'IN_PROGRESS'})


class SubmissionProgress:
    def __init__(self, data_root, work_dir, label, *, interval=15):
        self.root, self.work, self.label = Path(data_root), Path(work_dir), label
        self.interval, self.main_id = interval, threading.get_ident()
        self.started, self.last_saved = time.monotonic(), 0
        self.phase = 'Entering submission workflow'
        self.binding = {}
        self.stop = threading.Event()

    def location(self):
        frame = sys._current_frames().get(self.main_id)
        location = None
        while frame is not None:
            if frame.f_code.co_name in {'read_json', 'sha256_file'}:
                value = frame.f_locals.get('path')
                if isinstance(value, (str, Path)):
                    location = str(value)
                    break
            if 'vietmedbridge' in frame.f_code.co_filename:
                location = Path(frame.f_code.co_filename).name + ':' + str(frame.f_lineno)
            frame = frame.f_back
        return location

    def snapshot(self, state='RUNNING'):
        return {'state': state, 'variant': self.label, 'phase': self.phase,
            'elapsed_seconds': round(time.monotonic() - self.started, 1),
            'waiting_at': self.location(), 'updated_at': utc_now(),
            **self.binding,
            'scope': 'Live stage diagnostic; not an embedding completion or submission receipt'}

    def report(self, message):
        self.phase = message
        print(message, flush=True)
        if time.monotonic() - self.last_saved >= 15:
            try:
                atomic_json(self.root / 'retrieval/submission_progress.json', self.snapshot())
                self.last_saved = time.monotonic()
            except OSError:
                pass

    def heartbeat(self):
        while not self.stop.wait(self.interval):
            status = self.snapshot()
            print(f"SUBMISSION heartbeat | {status['elapsed_seconds']:.0f}s | {self.phase}"
                f" | waiting at {status['waiting_at']}", flush=True)
            try:
                atomic_json(self.work / 'submission_progress.json', status)
            except OSError:
                pass  # Diagnostics cannot replace or invalidate native checkpoint errors.


@contextmanager
def submission_gpu_progress(data_root, work_dir, *, label='baseline'):
    """Single-writer notebook adapter; restores every patched binding on exit."""
    from vietmedbridge import scale_baseline, submission_workflow
    progress = SubmissionProgress(data_root, work_dir, label)
    original_maps = scale_baseline.imported_bge_manifest
    names = ('_submission_context', '_saved_submission', '_release_catalog_staging', '_ensure_primary')
    originals = {name: getattr(submission_workflow, name) for name in names}

    def wrap(name, function):
        def observed(*args, **kwargs):
            progress.report('SUBMISSION stage: ' + name)
            result = function(*args, **kwargs)
            if name == '_submission_context':
                progress.binding = {'build_run': result[3].name,
                    'candidate_manifest_sha256': result[4]['candidate_manifest_sha256'],
                    'input_count': result[5]['input_count']}
            progress.report('SUBMISSION stage complete: ' + name)
            return result
        return observed

    scale_baseline.imported_bge_manifest = lambda *a, **k: read_bge_maps_parallel(*a, **k, progress=progress.report)
    for name, function in originals.items():
        setattr(submission_workflow, name, wrap(name, function))
    thread = threading.Thread(target=progress.heartbeat, name='submission-progress', daemon=True)
    progress.report('SUBMISSION startup | live progress every 15 seconds | same candidate and checkpoints')
    thread.start()
    state = 'STOPPED'
    try:
        yield progress
        state = 'CALL_FINISHED'
    finally:
        progress.stop.set()
        thread.join(timeout=1)
        scale_baseline.imported_bge_manifest = original_maps
        for name, function in originals.items():
            setattr(submission_workflow, name, function)
        status = progress.snapshot(state)
        try:
            atomic_json(Path(work_dir) / 'submission_progress.json', status)
        except OSError:
            pass
        try:
            atomic_json(Path(data_root) / 'retrieval/submission_progress.json', status)
        except OSError:
            pass


from vietmedbridge.submission_workflow import run_current_submission
STRONG = None
if GPU_SESSION:
    with submission_gpu_progress(DATA_ROOT, WORK_DIR, label="strong"):
        STRONG = run_current_submission(DATA_ROOT,CHECKOUT,WORK_DIR,code_commit=COMMIT)
    print(json.dumps(STRONG["ready"],ensure_ascii=False,indent=2),flush=True)
    print("Diagnostics:",STRONG["diagnostics_path"],flush=True)
    if STRONG["ready"]["state"] != "READY_FOR_MANUAL_UPLOAD":
        print("Bản strong chưa hoàn tất; baseline ZIP vẫn còn trên Drive. Run all cùng notebook để tiếp tục checkpoint.",flush=True)
else:
    print("GPU phase skipped on CPU.")


In [ ]:
#@title 5. Tải ZIP mới nhất đã kiểm tra (chạy cell này riêng cũng được)
from google.colab import files
latest = DATA_ROOT / "retrieval/latest_submission.json"
if latest.is_file():
    submission = read_json(latest)
    if submission["candidate_manifest_sha256"] != PREPARATION["candidate_manifest_sha256"]:
        raise ValueError("ZIP thuộc candidate khác; chưa tải nhầm kết quả cũ.")
    verify_file(submission["zip_path"],submission["zip_sha256"])
    print(json.dumps(submission,ensure_ascii=False,indent=2),flush=True)
    print("Upload submission.zip lên cổng BTC. URL cũ vẫn được giữ; chưa có official score cho lượt này.",flush=True)
    files.download(submission["zip_path"])
else:
    print("Chưa có ZIP hoàn tất. CPU preparation không phải submission; tiếp tục pha GPU.",flush=True)
